<a href="https://colab.research.google.com/github/kimmich001207-art/Kimmich/blob/main/03-LLM-Based%20News%20Headline%20CTR%20Prediction%20System/%20LLM_Based_News_Headline_CTR_Prediction_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Upworthy Case Study

---

## Learning Objectives

In this in-class exercise, you will learn:

1. **Zero-Shot Prompting**: How LLMs can make predictions without any training data
2. **Embedding-Based Prediction**: Using text embeddings + traditional ML for classification
3. **Fine-Tuning (SFT)**: Using a pre-trained fine-tuned model for improved predictions

---

## Background: The Upworthy Dataset

**Upworthy** was a viral content website known for A/B testing headlines. For each article, editors created multiple headline variants and tested which performed better with real users.

Our dataset contains **headline pairs** where:
- `headline_1` and `headline_2` are two versions tested for the same article
- `winner` indicates which headline performed better (1 or 2)

**Key Question**: Can we predict which headline will perform better?

This is a classic **preference learning** problem - we're not predicting a category, but learning which option humans prefer.

---
# Part 1: Environment Setup
---

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import time
import re
import ast
import asyncio
import warnings
warnings.filterwarnings('ignore')

# ML libraries
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit

# Google Colab data download
import gdown

# OpenAI (sync and async clients)
from openai import OpenAI, AsyncOpenAI
from getpass import getpass

In [ ]:
# Securely load your OpenAI API key
# This prevents your key from being saved in the notebook
api_key = getpass('Enter your OpenAI API key: ')

# Create both sync and async clients
client = OpenAI(api_key=api_key)
async_client = AsyncOpenAI(api_key=api_key)

print('OpenAI clients created successfully!')

Enter your OpenAI API key: ··········
OpenAI clients created successfully!


---
# Part 2: Load and Explore Data
---

In [ ]:
# Download headline pairs from Google Drive
file_id = "1q9joBnfcZaEkw_ZXs9opztbcEUr-STbP"
output_file = "winner.csv"

gdown.download(f"https://drive.google.com/uc?id={file_id}", output_file, quiet=False)
df = pd.read_csv(output_file)

print(f'Dataset shape: {df.shape}')
print(f'Unique articles (IDs): {df["ID"].nunique()}')
print(f'\nWinner distribution:\n{df["winner"].value_counts()}')

Downloading...
From: https://drive.google.com/uc?id=1q9joBnfcZaEkw_ZXs9opztbcEUr-STbP
To: /content/winner.csv
100%|██████████| 4.10M/4.10M [00:00<00:00, 37.5MB/s]


Dataset shape: (23682, 4)
Unique articles (IDs): 8624

Winner distribution:
winner
2    11906
1    11776
Name: count, dtype: int64


In [ ]:
# View sample data
df[df['ID'] == 10].head()


,ID,headline_1,headline_2,winner
22,10,You Have 50 Seconds To Disappear. Your Life De...,You Have 50 Seconds To Make A Suit Of Armor. GO!,1
23,10,You Have 50 Seconds To Disappear. Your Life De...,A Terrified Child Genius Saves His Own Life Wi...,2
24,10,A Terrified Child Genius Saves His Own Life Wi...,"I Wish I Had This Kid's Engineering Ability, B...",1
25,10,A Terrified Child Genius Saves His Own Life Wi...,You Have 50 Seconds To Make A Suit Of Armor. GO!,1


### Teaching Note: Understanding the Data Structure

- **ID**: Groups headlines for the same article (important for train/test splitting)
- **headline_1** and **headline_2**: The two headline variants being compared
- **winner**: Which headline performed better (1 or 2) based on user engagement metrics

Notice that the same headline can appear in multiple pairs (compared against different alternatives). This is why we need **group-based splitting** to avoid data leakage.

In [ ]:
# Let's look at one example in detail
sample_idx = 100
print("Example headline pair:")
print(f"\nHeadline 1: {df.iloc[sample_idx]['headline_1']}")
print(f"\nHeadline 2: {df.iloc[sample_idx]['headline_2']}")
print(f"\nWinner: Headline {df.iloc[sample_idx]['winner']}")

Example headline pair:

Headline 1: The Man Who Saved Children Destined For Concentration Camps Just Got Properly Thanked

Headline 2: 669 Jewish Children Were Saved From The Holocaust By A Single Man. This Is How They Thanked Him.

Winner: Headline 2


---
## Create a Consistent Test Set
---

To fairly compare all methods, we use the **same 300 test samples** across:
- Zero-shot prompting
- Embedding + Logistic Regression
- Fine-tuned model

In [ ]:
# Configuration: Same test set for all methods
N_TEST = 300      # Number of test samples
TEST_SEED = 42    # For reproducibility

# Create test set
df_test = df.sample(n=N_TEST, random_state=TEST_SEED).reset_index(drop=True)

print(f"Test set created: {len(df_test)} samples")
print(f"Winner distribution in test set: {df_test['winner'].value_counts().to_dict()}")

Test set created: 300 samples
Winner distribution in test set: {2: 167, 1: 133}


---
# Part 3: Zero-Shot Prompting
---

### Teaching Note: What is Zero-Shot Learning?

**Zero-shot learning** means asking a model to perform a task it wasn't explicitly trained for, without providing any examples. We simply describe what we want in a prompt.

**Advantages:**
- No training data needed
- Instant deployment
- Easy to iterate on prompts

**Limitations:**
- Performance depends heavily on prompt design
- May not capture domain-specific patterns
- Inconsistent results across different inputs

Let's test if GPT-4.1-mini can predict which headline performs better just by understanding the task.

We use OpenAI's **Responses API** (`client.responses.create`):

| Parameter | Type | Description |
|---|---|---|
| `model` | string | Which model to use (e.g., `gpt-4.1-mini-2025-04-14`) |
| `instructions` | string | System-level guidance (like a system message) |
| `input` | string | The user's query/task |
| `store` | bool | Whether to store the response for future training (set to False for evaluation) |

In [ ]:
# System prompt for headline comparison
SYSTEM_PROMPT = (
    "You are an expert at predicting which headlines will get more engagement (clicks, shares). "
    "You understand what makes content go viral: emotional appeal, curiosity gaps, specificity, and relevance."
)

def format_user_prompt(headline_1, headline_2):
    """Format the user prompt for headline comparison."""
    return (
        f"Which headline will get more clicks? Reply with ONLY the number (1 or 2).\n\n"
        f"1. {headline_1}\n\n"
        f"2. {headline_2}"
    )

def parse_prediction(output_text):
    """Parse the model's output to extract the predicted label (1 or 2)."""
    match = re.search(r'[12]', str(output_text).strip())
    return int(match.group()) if match else 1  # Default to 1 if parsing fails

print("Prompt functions defined.")

Prompt functions defined.


In [ ]:
async def predict_headline_async(headline_1, headline_2, sem, model_name="gpt-4.1-mini-2025-04-14"):
    """
    Async function to predict winning headline.
    Uses semaphore to limit concurrent requests.
    """
    user_prompt = format_user_prompt(headline_1, headline_2)

    async with sem:
        response = await async_client.responses.create(
            model=model_name,
            instructions=SYSTEM_PROMPT,
            input=user_prompt,
            store=False
        )

    return response.output_text

async def predict_all_headlines(df_samples, model_name="gpt-4.1-mini-2025-04-14", concurrency=50):
    """
    Predict all headline pairs concurrently.
    Returns list of raw output texts.
    """
    sem = asyncio.Semaphore(concurrency)
    tasks = [
        predict_headline_async(row['headline_1'], row['headline_2'], sem, model_name)
        for _, row in df_samples.iterrows()
    ]
    return await asyncio.gather(*tasks)

print("Async prediction functions defined.")

Async prediction functions defined.


In [ ]:
# Run zero-shot predictions on test set
print(f"Running zero-shot predictions on {len(df_test)} samples...")
print("(Using concurrent API calls for speed)\n")

t0 = time.perf_counter()
raw_outputs_zeroshot = await predict_all_headlines(df_test, model_name="gpt-4.1-mini-2025-04-14")
t1 = time.perf_counter()

# Parse predictions
df_test['pred_zeroshot_raw'] = raw_outputs_zeroshot
df_test['pred_zeroshot'] = df_test['pred_zeroshot_raw'].apply(parse_prediction)
df_test['correct_zeroshot'] = df_test['pred_zeroshot'] == df_test['winner']

# Calculate accuracy
acc_zeroshot = df_test['correct_zeroshot'].mean()

print(f"Completed in {t1 - t0:.1f} seconds")
print(f"\n{'='*50}")
print("ZERO-SHOT PROMPTING RESULTS")
print(f"{'='*50}")
print(f"\nCorrect predictions: {df_test['correct_zeroshot'].sum()} / {len(df_test)}")
print(f"Accuracy: {acc_zeroshot:.1%}")
print(f"\nNote: Random guessing would achieve ~50% accuracy")

Running zero-shot predictions on 300 samples...
(Using concurrent API calls for speed)

Completed in 7.0 seconds

ZERO-SHOT PROMPTING RESULTS

Correct predictions: 203 / 300
Accuracy: 67.7%

Note: Random guessing would achieve ~50% accuracy


In [ ]:
# View some predictions
df_test[['headline_1', 'headline_2', 'winner', 'pred_zeroshot', 'correct_zeroshot']].head(10)

,headline_1,headline_2,winner,pred_zeroshot,correct_zeroshot
0,A TV Star Has To Explain Why A White Man Killi...,A TV Star Explain's Why The Dunn Trial Is A Wh...,1,1,True
1,Before You Ask: No. Her Mother Did Not Put Her...,Why Should Gay People Be Together? Ask An 8-Ye...,2,2,True
2,"Why 'Hey, I Just Met Someone & Gave Them My Ph...",What A Guy Hopes For When He Asks For A Woman'...,2,2,True
3,The Guy Pouring Milk On His ‘Lady Flakes’ Is A...,Watch A Guy From The UK Make A Bowl Of Cereal ...,1,2,False
4,Why It’s BS To Tell Women To Dress Conservativ...,It’s Total BS To Try And Change Women’s Behavi...,2,2,True
5,They're Skipping School Because Of Their Perio...,They're Missing School Because They Can't Affo...,1,2,False
6,3 Generations Live Under The Same Roof. It's N...,A Woman Hears From Her Mother The 3 Words She ...,2,2,True
7,How To Tell If You Are Secretly A Feminist And...,The Not-So-Secret Feminist Agenda And How It C...,1,1,True
8,He Had Me At The Joke. But The Final Message? ...,The First Half I Was Laughing Out Loud. But By...,2,2,True
9,Fired From A Job Because... Religious Freedom?...,How A Teacher Got Fired For Being Gay When Her...,2,2,True


**Expected performance: ~60-70%**

The model has general knowledge about engaging content (emotional hooks, curiosity gaps, etc.) but hasn't learned the specific patterns that make Upworthy headlines successful.

**Why not higher?**
- Upworthy had a distinctive style that evolved over time
- What "works" depends on the specific audience and platform
- Subtle word choices can make big differences in engagement

---
# Part 4: Embedding-Based Prediction
---

**Embeddings** convert text into dense numerical vectors that capture semantic meaning. Similar texts have similar embeddings (close in vector space).

**Our approach:**
1. Get embeddings for both headlines using OpenAI's embedding model (you can define the dimension of embeddings). Here I extracted embeddings for you. You only need to download embeddings.
2. Concatenate the two embedding vectors
3. Train a logistic regression classifier to predict the winner

**Advantages over zero-shot:**
- Learns patterns from data
- Fast inference (no API calls after training)
- Interpretable (can analyze what features matter)

In [ ]:
# Download pre-computed embeddings (to save time/cost)
file_id = "1Qd5Sct8ozrcd53F0jQQNJCsBJrQnyyd4"
output_file = "winner_embedded.csv"

gdown.download(f"https://drive.google.com/uc?id={file_id}", output_file, quiet=False)
df_embedded = pd.read_csv(output_file)

print(f"Loaded {len(df_embedded)} samples with pre-computed embeddings")
df_embedded.head()

Downloading...
From (original): https://drive.google.com/uc?id=1Qd5Sct8ozrcd53F0jQQNJCsBJrQnyyd4
From (redirected): https://drive.google.com/uc?id=1Qd5Sct8ozrcd53F0jQQNJCsBJrQnyyd4&confirm=t&uuid=25c69432-9990-409b-aadc-df745a23aee7
To: /content/winner_embedded.csv
100%|██████████| 269M/269M [00:02<00:00, 91.6MB/s]


Loaded 23682 samples with pre-computed embeddings


,ID,headline_1,headline_2,winner,embedding_1,embedding_2
0,0,"SCIENCE FACT: Gay Science, Like Straight Scien...",If You Know Anyone Who Is Afraid Of Gay People...,2,"[0.10993286967277527, -0.022515302523970604, -...","[-0.013630201108753681, -0.04130033031105995, ..."
1,0,"Hey Dude. If You Have An Older Brother, There'...",If You Know Anyone Who Is Afraid Of Gay People...,2,"[-0.024494925513863564, -0.03634834662079811, ...","[-0.013630201108753681, -0.04130033031105995, ..."
2,0,"Here's The Science, Here's The Gay. Open Your ...",If You Know Anyone Who Is Afraid Of Gay People...,2,"[0.09171925485134125, -0.06546857208013535, -0...","[-0.013630201108753681, -0.04130033031105995, ..."
3,0,I've Got Some News For You. Being Gay Is Genet...,If You Know Anyone Who Is Afraid Of Gay People...,2,"[-0.014668514020740986, -0.045854512602090836,...","[-0.013630201108753681, -0.04130033031105995, ..."
4,1,Why Yoko Ono Is The Only Thing Standing Betwee...,New York's Last Chance To Preserve Its Water S...,1,"[0.01985621638596058, -0.21701404452323914, -0...","[-0.08053235709667206, -0.1099352315068245, 0...."


### Data Splitting Strategy

**Why group-based splitting?**

Headlines from the same article (same ID) must stay in the same split. Otherwise:
- Training data might contain a headline that also appears in test data
- This causes **data leakage** - artificially inflated test performance

We use `GroupShuffleSplit` to ensure proper separation.

In [ ]:
def dataset_extraction(df, test_size=0.2, rs=42):
    """
    Split data ensuring no headline overlap between train and test.
    """
    # Group-based split by article ID
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=rs)
    train_idx, test_idx = next(gss.split(df, groups=df['ID']))
    train_df = df.iloc[train_idx]
    test_df_original = df.iloc[test_idx]

    # Additional filtering: remove test samples with headlines seen in training
    train_headlines = set(train_df['headline_1'].tolist() + train_df['headline_2'].tolist())
    test_df = test_df_original[~test_df_original['headline_1'].isin(train_headlines)]
    test_df = test_df[~test_df['headline_2'].isin(train_headlines)]

    print(f'Training samples: {len(train_df)}')
    print(f'Test samples (after filtering): {len(test_df)}')
    print(f'Filtered out {len(test_df_original) - len(test_df)} overlapping samples')

    # Convert embeddings from string to numpy arrays
    embed_1 = df['embedding_1'].apply(lambda x: np.array(ast.literal_eval(x)))
    embed_2 = df['embedding_2'].apply(lambda x: np.array(ast.literal_eval(x)))

    # Create feature matrices (concatenated embeddings)
    X_train = np.concatenate([
        np.stack(embed_1[train_df.index].tolist()),
        np.stack(embed_2[train_df.index].tolist())
    ], axis=1)
    y_train = train_df['winner'].values.astype(np.float32) - 1  # Convert 1,2 to 0,1

    X_test = np.concatenate([
        np.stack(embed_1[test_df.index].tolist()),
        np.stack(embed_2[test_df.index].tolist())
    ], axis=1)
    y_test = test_df['winner'].values.astype(np.float32) - 1

    return X_train, y_train, X_test, y_test, test_df

# Split the data
X_train, y_train, X_test_embed, y_test_embed, df_test_embed = dataset_extraction(df_embedded, test_size=0.2)

Training samples: 18867
Test samples (after filtering): 3692
Filtered out 1123 overlapping samples


In [ ]:
# Train logistic regression classifier
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(X_train, y_train)

# Evaluate on embedding test set
train_acc = clf.score(X_train, y_train)
test_acc_embed = clf.score(X_test_embed, y_test_embed)

print(f"{'='*50}")
print("EMBEDDING + LOGISTIC REGRESSION RESULTS")
print(f"{'='*50}")
print(f"\nTraining accuracy: {train_acc:.1%}")
print(f"Test accuracy: {test_acc_embed:.1%}")
print(f"\nNote: This model learned from {len(X_train)} training examples")

EMBEDDING + LOGISTIC REGRESSION RESULTS

Training accuracy: 82.0%
Test accuracy: 81.3%

Note: This model learned from 18867 training examples


Now let's evaluate on the **consistent subset** of the test set for fair comparison:

In [ ]:
# Match our test set with embedded data
# We need to find the same samples in df_embedded

# Create a lookup key for matching (strip to reduce whitespace mismatches)
df_test['lookup_key'] = df_test['headline_1'].str.strip() + '|||' + df_test['headline_2'].str.strip()
df_embedded['lookup_key'] = df_embedded['headline_1'].str.strip() + '|||' + df_embedded['headline_2'].str.strip()

# Ensure unique embedded keys
df_embedded_unique = df_embedded.drop_duplicates(subset=['lookup_key'], keep='first')

# Find matching samples (consistent subset)
df_test_with_embed = df_test.merge(
    df_embedded_unique[['lookup_key', 'embedding_1', 'embedding_2']],
    on='lookup_key',
    how='inner',
    validate='many_to_one'
)

df_test_consistent = df_test[df_test['lookup_key'].isin(df_test_with_embed['lookup_key'])].copy()
coverage = len(df_test_consistent) / len(df_test) if len(df_test) else 0
print(f"Matched {len(df_test_consistent)} / {len(df_test)} test samples with embeddings (coverage: {coverage:.1%})")

# Predict on matched samples
if len(df_test_with_embed) > 0:
    embed_1 = df_test_with_embed['embedding_1'].apply(lambda x: np.array(ast.literal_eval(x)))
    embed_2 = df_test_with_embed['embedding_2'].apply(lambda x: np.array(ast.literal_eval(x)))

    X_test_consistent = np.concatenate([
        np.stack(embed_1.tolist()),
        np.stack(embed_2.tolist())
    ], axis=1)

    # Predict (0/1) and convert back to (1/2)
    preds_embed = clf.predict(X_test_consistent).astype(int) + 1
    df_test_with_embed['pred_embedding'] = preds_embed
    df_test_with_embed['correct_embedding'] = df_test_with_embed['pred_embedding'] == df_test_with_embed['winner']

    acc_embedding = df_test_with_embed['correct_embedding'].mean()
    print(f"\nEmbedding accuracy on consistent test set: {acc_embedding:.1%}")

    # Fair comparison: evaluate zero-shot on same subset if available
    if 'pred_zeroshot' in df_test_consistent.columns:
        acc_zeroshot_consistent = (df_test_consistent['pred_zeroshot'] == df_test_consistent['winner']).mean()
        print(f"Zero-shot accuracy on consistent test set: {acc_zeroshot_consistent:.1%}")
    else:
        acc_zeroshot_consistent = acc_zeroshot
        print("Zero-shot predictions not found yet; using full test accuracy.")
else:
    acc_embedding = test_acc_embed  # fallback; should not happen
    acc_zeroshot_consistent = acc_zeroshot
    print(f"\nUsing full test set accuracy: {acc_embedding:.1%}")

Matched 300 / 300 test samples with embeddings (coverage: 100.0%)

Embedding accuracy on consistent test set: 81.3%
Zero-shot accuracy on consistent test set: 67.7%


### Why Embeddings Work Well Here

The embedding approach achieves **~79-82% accuracy** - significantly better than zero-shot prompting. Why?

1. **Data-driven**: Learns from 19,000+ labeled examples
2. **Pattern extraction**: Logistic regression finds linear combinations of embedding features that predict winners
3. **Domain-specific**: Captures what makes Upworthy headlines specifically engaging, not just general content principles

**Trade-off**: Requires labeled training data (which may not always be available)

---
# Part 5: Fine-Tuning (SFT)
---

### Teaching Note: What is Fine-Tuning?

**Supervised Fine-Tuning (SFT)** trains a model to produce specific outputs for given inputs. Unlike zero-shot prompting, the model learns from examples.

In **Assignment 4**, you will learn about SFT for customer service responses. Here, we apply the same concept to headline selection.

**Key difference from embeddings:**
- Embeddings + ML: Model provides features; separate classifier makes decisions
- Fine-tuning: Model itself learns to make the decisions

### How Fine-Tuning Works

1. **Prepare training data** in JSONL format with input-output pairs
2. **Upload** the training file to OpenAI
3. **Create a fine-tuning job** specifying the base model
4. **Wait** for training to complete (range from a few minutes to a few hours depending on tasks)
5. **Use** the fine-tuned model with the same API

### Fine-Tuning Training Data Format (SFT)

For SFT, we use the standard `messages` format:

```json
{"messages": [
  {"role": "system", "content": "You are an expert at predicting which headlines..."},
  {"role": "user", "content": "Which headline will get more clicks?\n\n1. [A]\n\n2. [B]"},
  {"role": "assistant", "content": "2"}
]}
```

The model learns to produce the correct winner (1 or 2) for each headline pair.

### Fine-Tuning Code (Reference Only)

The code below shows how to fine-tune a model. **DO NOT RUN THIS** - it costs money and takes time.

We have already fine-tuned a model that you can use for evaluation.

---

**Step 1: Create training data**
```python
# Create JSONL training file
training_data = []
for _, row in df_train.iterrows():
    user_msg = format_user_prompt(row['headline_1'], row['headline_2'])
    example = {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": str(row['winner'])}
        ]
    }
    training_data.append(example)

# Save to JSONL
with open('sft_training_data.jsonl', 'w') as f:
    for example in training_data:
        f.write(json.dumps(example) + '\n')
```

**Step 2: Upload and create fine-tuning job**
```python
# Upload training file
file = client.files.create(
    file=open('sft_training_data.jsonl', 'rb'),
    purpose='fine-tune'
)

# Create fine-tuning job
job = client.fine_tuning.jobs.create(
    training_file=file.id,
    model="gpt-4o-2024-08-06"
)

# Monitor at: https://platform.openai.com/finetune
```

**Step 3: Monitor until complete**
```python
while True:
    job = client.fine_tuning.jobs.retrieve(job.id)
    if job.status in ['succeeded', 'failed', 'cancelled']:
        break
    time.sleep(30)

fine_tuned_model = job.fine_tuned_model
```

### Use the Pre-Trained Fine-Tuned Model

We have already fine-tuned a model on headline data. You can use it with the **same Responses API** you've been using:

**Demo (do not run in class)**
```python
# Pre-trained fine-tuned model (already trained - you can use this directly!)
FINE_TUNED_MODEL = "ft:gpt-4o-2024-08-06:uw-mib-llm::AImiGZwJ"

print(f"Using pre-trained fine-tuned model: {FINE_TUNED_MODEL}")
print("\nThis model was fine-tuned on headline data using SFT.")
print("You can use it with the same Responses API!")
```

**Output (instructor run)**
```
Using pre-trained fine-tuned model: ft:gpt-4o-2024-08-06:uw-mib-llm::AImiGZwJ

This model was fine-tuned on headline data using SFT.
You can use it with the same Responses API!
```


**Demo (do not run in class)**
```python
# Test the fine-tuned model on one example
test_row = df_test.iloc[0]

response = client.responses.create(
    model=FINE_TUNED_MODEL,
    instructions=SYSTEM_PROMPT,
    input=format_user_prompt(test_row['headline_1'], test_row['headline_2']),
    store=False
)

print("Testing fine-tuned model on one example:")
print(f"\nHeadline 1: {test_row['headline_1'][:60]}...")
print(f"Headline 2: {test_row['headline_2'][:60]}...")
print(f"\nActual winner: {test_row['winner']}")
print(f"Model prediction: {response.output_text}")
```

**Output (instructor run)**
```
Testing fine-tuned model on one example:

Headline 1: A TV Star Has To Explain Why A White Man Killing A Black Kid...
Headline 2: A TV Star Explain's Why The Dunn Trial Is A White, American,...

Actual winner: 1
Model prediction: 2
```


**Demo (do not run in class)**
```python
# Evaluate fine-tuned model on all test samples
print(f"Evaluating fine-tuned model on {len(df_test)} samples...")
print("(Using concurrent API calls for speed)\n")

t0 = time.perf_counter()
raw_outputs_sft = await predict_all_headlines(df_test, model_name=FINE_TUNED_MODEL, concurrency=50)
t1 = time.perf_counter()

# Parse predictions
df_test['pred_sft_raw'] = raw_outputs_sft
df_test['pred_sft'] = df_test['pred_sft_raw'].apply(parse_prediction)
df_test['correct_sft'] = df_test['pred_sft'] == df_test['winner']

# Calculate accuracy
acc_sft = df_test['correct_sft'].mean()

print(f"Completed in {t1 - t0:.1f} seconds")
print(f"\n{'='*50}")
print("FINE-TUNED MODEL (SFT) RESULTS")
print(f"{'='*50}")
print(f"\nCorrect predictions: {df_test['correct_sft'].sum()} / {len(df_test)}")
print(f"Accuracy: {acc_sft:.1%}")

# Consistent-set accuracy (fair comparison with embeddings)
if 'df_test_consistent' in globals() and 'lookup_key' in df_test.columns and len(df_test_consistent) > 0:
    consistent_keys = set(df_test_consistent['lookup_key'])
    df_tmp = df_test[df_test['lookup_key'].isin(consistent_keys)]
    acc_sft_consistent = (df_tmp['pred_sft'] == df_tmp['winner']).mean()
    print(f"Fine-tuned accuracy on consistent test set: {acc_sft_consistent:.1%}")
else:
    acc_sft_consistent = acc_sft
    print("Consistent subset not available; using full test accuracy.")
```

**Output (instructor run)**
```
Evaluating fine-tuned model on 300 samples...
(Using concurrent API calls for speed)

Completed in 38.3 seconds

==================================================
FINE-TUNED MODEL (SFT) RESULTS
==================================================

Correct predictions: 244 / 300
Accuracy: 81.3%
Fine-tuned accuracy on consistent test set: 81.3%
```


---
# Part 6: Final Comparison (Reference Only)
---

```python
# Display comparison results (consistent subset)
acc_zeroshot_final = acc_zeroshot_consistent if 'acc_zeroshot_consistent' in globals() else acc_zeroshot
acc_sft_final = acc_sft_consistent if 'acc_sft_consistent' in globals() else acc_sft
acc_embedding_final = acc_embedding

if 'df_test_consistent' in globals():
    n_consistent = len(df_test_consistent)
    coverage = n_consistent / len(df_test) if len(df_test) else 0
    subset_info = f"{n_consistent} samples (coverage {coverage:.1%})"
else:
    subset_info = f"{N_TEST} samples"

print(f"\n{'='*60}")
print("MODEL COMPARISON RESULTS (CONSISTENT SUBSET)")
print(f"{'='*60}")
print(f"\nTest set size: {subset_info}")
print(f"\n{'Model':<35} {'Accuracy':>10}")
print("-" * 47)
print(f"{'Random Baseline':<35} {'50.0%':>10}")
print(f"{'Zero-Shot (gpt-4.1-mini)':<35} {f'{acc_zeroshot_final:.1%}':>10}")
print(f"{'Embedding + LogReg (19K examples)':<35} {f'{acc_embedding_final:.1%}':>10}")
print(f"{'Fine-Tuned SFT (gpt-4o)':<35} {f'{acc_sft_final:.1%}':>10}")
```

**Output (instructor run)**
```
============================================================
MODEL COMPARISON RESULTS (CONSISTENT SUBSET)
============================================================

Test set size: 300 samples (coverage 100.0%)

Model                                 Accuracy
-----------------------------------------------
Random Baseline                          50.0%
Zero-Shot (gpt-4.1-mini)                 67.7%
Embedding + LogReg (19K examples)        81.3%
Fine-Tuned SFT (gpt-4o)                  81.3%
```

---
# Summary
---

In this exercise, we explored three approaches to headline selection:

| Approach | Accuracy | Training Data | Cost | Best For |
|----------|----------|---------------|------|----------|
| Zero-shot | ~67% | None | Per-query only | Quick prototyping |
| Embedding + ML | ~81% | ~19,000 | Embedding cost | Fast inference |
| Fine-tuned SFT | ~81% | ~19,000 | Training + per-query | Maximum accuracy |

1. **Zero-shot** is less capable because limited by lack of domain knowledge
2. **Fine-tuning** lets you teach the model domain-specific patterns
3. **Embeddings + traditional ML** are a strong alternative when Y in your task is structured
4. **Same API** - The Responses API works with both base models and fine-tuned models.

## SFT v.s. Embeddings+ML

**SFT is better when the target \(Y\) is *free-form generated text*, rather than a low-dimensional structured label.**

---

## Embedding + ML: Best Use Cases

**Use when \(Y\) is:**
- Discrete labels (classification)
- Continuous scalars (regression)
- Low-dimensional structured outputs

**Typical tasks:**
- Sentiment classification  
- Topic/category prediction  
- CTR / conversion prediction  
- Rating or score prediction  
- Binary relevance / ranking

**Pipeline:**
- `X → Embedding Model → ML Head → Y`
- Learn a discriminative mapping from representations to labels

---

## SFT (Supervised Fine-Tuning): Best Use Cases

**Use when \(Y\) is:**
- High-dimensional, free-form text
- Structured language outputs with format, tone, or style constraints

**Typical tasks:**
- Text generation (summarization, rewriting)
- Explanation / rationale generation
- SQL / code generation
- Dialogue / agent responses
- Report, email, or content drafting

**Pipeline:**
- `X → LLM (fine-tuned end-to-end) → Y`
- Learn the full conditional distribution \(p(Y \mid X)\)

---


> Embedding + ML is for *predicting labels*; SFT is for *generating text*.

> Embedding + ML learns a discriminative mapping \(f(\phi(X)) \rightarrow Y\), while SFT adapts the generative distribution \(p(Y \mid X)\), which is necessary when \(Y\) is complex, structured language.